In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from acc_sync import util, viz

import numpy as np
import pandas as pd

# 1: Load data, view dataframe, and plot

In [ ]:
# load the session data
session = util.load_session_data(
    pt="RF001",
    session="2025_12_13",
    roles=["client", "therapist"],
    hands=["R"] # use ["R", "L"] for both hands
)

# We trim the session to the time of interest, fill missing values, and filter with a 10 Hz lowpass and 0.1 Hz highpass filter
acc_mag = util.prep_session_mag(session, start_time='15:04:50', end_time='15:32:42')

# Load the (mock) session notes and anchor them to the recording date so they
# line up with the signal panels on the shared time axis.
notes = pd.read_csv("data/RF001/2025_12_13/session_notes.csv")
session_date = "2025-12-13"

In [ ]:
# We can plot the data using interactive plotly figures.
for wearable in session:
    viz.plot_acc(session[wearable]).update_layout(
        title=f"Accelerometer Data: {wearable}"
    ).show()

viz.plot_mag_dict(acc_mag).show()
# Click legend to toggle visibility in cases of overlap.

# 2: Time domain analysis of synchrony via Time-Lag Cross-Correlation

In [ ]:
# For each wearable pair, show the session notes, the magnitudes, the windowed
# cross-correlation, and the max correlation stacked on one shared time axis.
x = "therapist_R"
y = "client_R"

r_sq = util.frame_xcorr(acc_mag[x], acc_mag[y]) ** 2
fig = viz.combine_time_figs(
    [
        viz.plot_mag_dict({x: acc_mag[x], y: acc_mag[y]}),
        viz.plot_corr_df(r_sq),
        viz.plot_line(r_sq.max(axis=1), name="max r", yaxis_title="Max r", yrange=[0, 1]),
        viz.plot_session_notes(notes, date=session_date),
    ],
    titles=[
        f"Magnitude: {x} vs {y}", 
        "Windowed Cross Correlation", 
        "Max Correlation", 
        "Session Notes"
    ],
    row_heights=[1, 2, 1, 0.4],
)
fig.show()

# 3 Time-Frequency based synchrony analysis method: 
- using STFT as PoC for now, but literature suggests wavelet transform.
- Other Time-Frequency representations may include: CQT, HHT, etc.
- Besides coherence, other measures include: Phase Locking Value (PLV), Phase Lag Index (PLI), etc.

In [ ]:
# Compute spectrograms for each wearable and visualize them.
for wearable in acc_mag:
    S = util.stft(acc_mag[wearable], win_length=5.12, hop=0.5, sr=25.0, verbose=True)
    viz.plot_spec(S).update_layout(title=f"{wearable} Spectrogram Magnitude").show()

In [ ]:
# Cross spectrum S_i * S_j^* and magnitude-squared coherence for each pair,
# stacked with the session notes and average summaries on one shared time axis.
x = "therapist_R"
y = "client_R"

S_i = util.stft(acc_mag[x], win_length=5.12, hop=0.5, sr=25.0)
S_j = util.stft(acc_mag[y], win_length=5.12, hop=0.5, sr=25.0)
# Multiply positionally (not by datetime label) and keep S_i's time axis
S_ij = pd.DataFrame(
    S_i.values * np.conj(S_j.values), index=S_i.index, columns=S_i.columns,
)

# # Physical smoothing has large effects on the resulting coherence; tune as needed.
# sigma_f, sigma_t = util.physical_to_bin_sigmas(2, 0.5, 25.0, 128, 12)
# coherence = util.spec_coherence(S_i, S_j, sigma=(sigma_f, sigma_t))

# Mean cross-spectral energy per frame, in dB relative to a fixed 1 g^2
# reference (stable across sessions, unlike a per-session dBFS scale).
avg_energy = util.mean_cross_energy(S_ij, db=False)
# avg_coherence = coherence.iloc[5:-20, :].mean(axis=0)

fig = viz.combine_time_figs(
    [
        viz.plot_spec(S_ij),
        viz.plot_line(avg_energy, name="mean cross energy", yaxis_title="Energy in g²"),
        # viz.plot_coherence(coherence),
        # viz.plot_line(avg_coherence, name="mean coherence", yaxis_title="Coherence", yrange=[0, 1]),
        viz.plot_session_notes(notes, date=session_date),
    ],
    titles=[
        f"{x} vs {y} Cross Spectrum", "Mean Cross-Spectral Energy",
        # "Magnitude-Squared Coherence", "Mean Coherence",
        "Session Notes",
    ],
    row_heights=[2, 1, 0.4],
)
fig.show()

# End of notebook